### Bulls of Kathmandu

Relational database project for NEPSE analytics, integrating historical data (2011–2025) to support efficient querying and financial reporting.

### Renaming required column to load into SQL for company_list.csv

In [23]:
import uuid
import pandas as pd

df = pd.read_csv("./Data/company_list.csv")
df['id'] = [str(uuid.uuid4()) for _ in range(len(df))]
df.columns = df.columns.str.lower()
print(df.columns.tolist())

['sector', 'serial_number', 'symbol_link', 'symbol_text', 'symbol', 'symbol_1', 'company_link', 'company_text', 'company', 'listed_share', 'paid_up_rs', 'total_paid_up_capital_rs', 'market_capitalization_rs', 'date_of_operation', 'ltp', 'as_of', 'id']


In [24]:
df = df.rename(columns={
    's.n.': 'serial_number',
    'listed share': "listed_share",
    "paid-up (rs)": "paid_up_rs",
    'total paid-up capital (rs)': "total_paid_up_capital_rs",
    "market capitalization (rs)": "market_capitalization_rs",
    "date of operation": "date_of_operation",
    "as of:": "as_of",
    "symbol.1": "symbol_1",
})
print(df.columns.tolist())

['sector', 'serial_number', 'symbol_link', 'symbol_text', 'symbol', 'symbol_1', 'company_link', 'company_text', 'company', 'listed_share', 'paid_up_rs', 'total_paid_up_capital_rs', 'market_capitalization_rs', 'date_of_operation', 'ltp', 'as_of', 'id']


### Let's export this normalized columns as CSV.

In [25]:
import os

export_path = "./Data/company_list.csv"
if os.path.exists(export_path): # basically we need to override it now
    os.remove(export_path)
df.to_csv(export_path, index=False)
company_list_df = df

### Renaming required column to load into SQL for combined_price_history.csv

In [26]:
df = pd.read_csv("./Data/combined_price_history.csv")
df['id'] = [str(uuid.uuid4()) for _ in range(len(df))]
df.columns = df.columns.str.lower()
print(df.columns.tolist())
combined_price_history_df = df

['symbol', 'company_name', 'sector', 'symbol_link', 'serial_number', 'date_added', 'open', 'high', 'low', 'ltp', 'percent_change', 'qty', 'turnover', 'id']


In [27]:
df = df.rename(columns={
    's.n.': 'serial_number',
    '% change': 'percent_change',
    'date': "date_added",
    'open': 'unlocked'
})
print(df.columns.tolist())

['symbol', 'company_name', 'sector', 'symbol_link', 'serial_number', 'date_added', 'unlocked', 'high', 'low', 'ltp', 'percent_change', 'qty', 'turnover', 'id']


In [28]:
export_path = "./Data/combined_price_history.csv"
if os.path.exists(export_path): os.remove(export_path)
df.to_csv(export_path, index=False)

### In order to assign a primary key we need to check if they are unqiue.

In [29]:
df["serial_number"].is_unique

False

In [30]:
print(combined_price_history_df.columns.tolist())

['symbol', 'company_name', 'sector', 'symbol_link', 'serial_number', 'date_added', 'open', 'high', 'low', 'ltp', 'percent_change', 'qty', 'turnover', 'id']


In [31]:
print(company_list_df.columns.tolist())

['sector', 'serial_number', 'symbol_link', 'symbol_text', 'symbol', 'symbol_1', 'company_link', 'company_text', 'company', 'listed_share', 'paid_up_rs', 'total_paid_up_capital_rs', 'market_capitalization_rs', 'date_of_operation', 'ltp', 'as_of', 'id']


## LETS RUN THAT LOCAL QUERY USING PYTHON TO CREATE TABLE IN SQL

In [32]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [37]:
import os
import psycopg2
from dotenv import load_dotenv

load_dotenv()
connection_string = os.getenv('CONNECTION_STRING')

conn = psycopg2.connect(connection_string)
cur = conn.cursor()


In [38]:

with open("Schema/company.sql", "r") as file:
    create_table = file.read()

''' Execute that create table for me '''
with conn.cursor() as cursor:
    cursor.execute(create_table)


''' Create table combined_price_history '''
with open("Schema/combined_price_history.sql", "r") as file:
    create_table_combined_price_history = file.read()

''' Execute combined table '''
with conn.cursor() as cursor:
    cursor.execute(create_table_combined_price_history)


conn.commit()